In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch

import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset,DataLoader

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
device="cuda" if torch.cuda.is_available() else "cpu"

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


        

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#load data
df=pd.read_csv("/kaggle/input/digit-recognizer/train.csv")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, train_size=0.8, random_state=42)
train_df = train_df.to_numpy()
test_df = test_df.to_numpy()

y_train, x_train = train_df[:, 0], train_df[:, 1:].astype(np.float32)
y_test, x_test = test_df[:, 0], test_df[:, 1:].astype(np.float32)

# normalize pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0


In [ ]:
x_train, y_train, x_test, y_test = map(torch.tensor, (x_train, y_train, x_test, y_test))


In [ ]:

eps = 1e-6
def relu(x):
    return x.clamp(min=0.) #output 0 if negative else x

def batch_norm(x, gamma): #normalise the data
    batch_mean = x.mean(dim=0)
    batch_var = x.var(dim=0) 
    inv_var = (batch_var + eps).pow(-.5)
    x_norm = (x-batch_mean) * inv_var
    return gamma * x_norm
batch_norm(torch.randn(10,5), 2)


In [ ]:

BATCH_SIZE=64
epochs=100
EPOCHS_PER_EVAL=1
lr = 1e-2
hidden_features=512

In [ ]:
#ensure all computation  happens on the same device 
x_train,y_train,x_test,y_test=[t.to(device) for t in (x_train,y_train,x_test,y_test)]

#Each row of w corresponds to weights connecting all 784 inputs to one neuron.
#one bias per neuron

w1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, 784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, )).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((hidden_features,hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((hidden_features,)).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((10,hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((10,)).to(device).requires_grad_()

In [ ]:

def forward(x, w1, b1, w2, b2, w3, b3):
    z1 = x @ w1.T + b1
    a1 = relu(z1)
    z2 = a1 @ w2.T + b2
    a2 = relu(z2)
    z3 = a2 @ w3.T + b3
    #apply softmax -> convert raw scores to probabilities 
    return torch.softmax(z3, dim=1)





In [ ]:
# fucntion to calculate loss 
def cross_entropy_loss(y_true,y_pred):
    #z3-> y predicted 
    p_true = y_pred[torch.arange(len(y_true)), y_true]
    loss = -torch.mean(torch.log(p_true + 1e-9))
    return loss
    


In [ ]:
#function to update weights and bias  
def update(lr, w1, b1, w2, b2, w3, b3):
    with torch.no_grad():
        w1 -= lr * w1.grad
        b1 -= lr * b1.grad
        w2 -= lr * w2.grad
        b2 -= lr * b2.grad
        w3 -= lr * w3.grad
        b3 -= lr * b3.grad
        
    #zero out gradients for next iteration
    w1.grad.zero_()
    b1.grad.zero_()
    w2.grad.zero_()
    b2.grad.zero_()
    w3.grad.zero_()
    b3.grad.zero_()


    

In [ ]:
#evaluate 
def evaluate(y_pred,y_train):
    with torch.no_grad(): 
        preds = torch.argmax(y_pred, dim=1) 
        correct = (preds == y_train).sum().item()
        total = y_train.size(0)
    return correct / total

In [ ]:
#loop for actual computations 
from torch.utils.data import TensorDataset, DataLoader
train_data = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(epochs):
    for xb, yb in train_loader:  # xb=batch of images, yb=batch of labels
        y_pred = forward(xb, w1, b1, w2, b2, w3, b3)
        loss = cross_entropy_loss(yb, y_pred)
        loss.backward()
        update(lr, w1, b1, w2, b2, w3, b3)

    # check train accuracy after each epoch
    with torch.no_grad():
        y_pred = forward(x_train, w1, b1, w2, b2, w3, b3)
        acc = evaluate(y_pred, y_train)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | Accuracy: {acc*100:.2f}%")
     
    

    
